# Stage E — Factorial rerun with a matched (light) Transformer block (POST HOC)

**Why:** the original factorial confounded task sharing with regularization. The ST arms used the light block (dropout only inside attention and once in the feed-forward path: **3 Dropout layers**), whereas the MTL arms used a heavier block (extra dropout after attention and after each feed-forward layer: **7 Dropout layers**). This notebook reruns **MTL-GAP and MTL-LSTA with the light block** so that the four arms differ only in task sharing and temporal aggregation.

**Protocol (identical to the factorial in the paper):** frozen German datasets (1, 6, 24 h), Adam lr 1e-3, batch 64, ≤50 epochs, early stopping patience 7 with best-weight restoration, MSE with load:solar weights 2:1, seeds 1, 7, 21, 42, 100. Parameters: 71,810 (unchanged — dropout has no weights).

**Options:** `RUN_ST_ARMS = True` also reruns ST-GAP and ST-LSTA in this same software environment (removes the software-version confound between arms; +60 network trainings). E4 is an optional reproducibility probe (run it before E5 so its result is bundled).

**Writes only to** `Benchmark_V5/IJGE_Revision/StageE_light_factorial/`. **Resumable:** completed runs are skipped. Runtime on A100: about 1.5–3 h for the 30 MTL runs (≈ +3 h with ST arms).

In [ ]:
# ===================== E0 — SETUP =====================
from google.colab import drive
drive.mount('/content/gdrive')
import os, sys, json, hashlib, gc, time, random, zipfile, platform
from pathlib import Path
import numpy as np, pandas as pd
MYDRIVE = Path('/content/gdrive/MyDrive'); BENCH = MYDRIVE / 'Benchmark_V5'
OUT = BENCH / 'IJGE_Revision' / 'StageE_light_factorial'; OUT.mkdir(parents=True, exist_ok=True)
SEEDS = [1, 7, 21, 42, 100]; HORIZONS = [1, 6, 24]
RUN_ST_ARMS = False          # set True to also rerun ST-GAP and ST-LSTA in this environment (recommended if time allows)
MAX_EPOCHS, PATIENCE, LR, BATCH = 50, 7, 1e-3, 64
TARGET_MEAN  = np.array([55821.81814021, 4184.23949544]); TARGET_SCALE = np.array([10034.61850806, 6415.36095093])
DATA_PATH = {1: BENCH / 'data' / 'frozen_168h_sequences.npz', 6: BENCH / 'MultiHorizon_V6' / 'datasets' / 'frozen_168h_h6_v6.npz',
             24: BENCH / 'MultiHorizon_V6' / 'datasets' / 'frozen_168h_h24_v6.npz'}
DATA_SHA = {1: 'ed67feaed54517cac5f84d314caca75a7b3837ed9bb7f17c7613f0023bd69912', 6: '8b87dd4eac8f8690e8f800b35a3549327783f8886deb89e285511f4c32b5653a',
            24: 'c4932ec4bda1b0608ac38a4456ccdc896059c281b90d51d9ee631e92eed55936'}
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
def guard(p):
    p = Path(p).resolve(); assert str(p).startswith(str(OUT.resolve())), f'refusing to write outside OUT: {p}'
    p.parent.mkdir(parents=True, exist_ok=True); return p
rmse = lambda e: float(np.sqrt(np.mean(np.asarray(e, np.float64) ** 2)))
import tensorflow as tf
for g in tf.config.list_physical_devices('GPU'): tf.config.experimental.set_memory_growth(g, True)
ENV = {'tensorflow': tf.__version__, 'python': platform.python_version(), 'gpu': [d.name for d in tf.config.list_physical_devices('GPU')]}
print(ENV, '| write location:', OUT)


In [ ]:
# ===================== E1 — DATA (hash-gated frozen German datasets) =====================
DATA = {}
for h in HORIZONS:
    got = sha256(DATA_PATH[h]); assert got == DATA_SHA[h], f'dataset hash mismatch h={h}: {got}'
    with np.load(DATA_PATH[h], allow_pickle=False) as z:
        DATA[h] = {k: np.asarray(z[k], np.float32) for k in ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test']}
    d = DATA[h]; assert d['X_test'].shape == (6576, 168, 8) and d['X_val'].shape[0] == 8760
    print(f'h={h:2d}: train {len(d["X_train"])}, val {len(d["X_val"])}, test {len(d["X_test"])} | SHA PASS')


In [ ]:
# ===================== E2 — BUILDERS + ARCHITECTURE AUDIT =====================
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class PositionalEncoding(layers.Layer):          # fixed sinusoidal encoding (no trainable weights), as in Step 10D
    def __init__(self, sequence_length=168, d_model=64, **kw):
        super().__init__(**kw)
        pos = np.arange(sequence_length)[:, None]; div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = np.zeros((sequence_length, d_model), np.float32); pe[:, 0::2] = np.sin(pos * div); pe[:, 1::2] = np.cos(pos * div)
        self.pe = tf.constant(pe[None], tf.float32)
    def call(self, x): return x + self.pe[:, :tf.shape(x)[1], :]

class LastState(layers.Layer):                    # LSTA: final contextual state (replaces the Lambda; no weights)
    def call(self, x): return x[:, -1, :]

def heavy_block(x, p):                            # MTL block used in the original runs (Step 5B / 10D): 3 Dropout layers
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.2, name=f'{p}_mha')(x, x)
    a = layers.Dropout(0.2, name=f'{p}_attn_dropout')(a)
    x = layers.LayerNormalization(epsilon=1e-6, name=f'{p}_attn_ln')(layers.Add()([x, a]))
    f = layers.Dense(128, activation=tf.keras.activations.gelu, name=f'{p}_ff1')(x)
    f = layers.Dropout(0.2, name=f'{p}_ff_dropout1')(f)
    f = layers.Dropout(0.2, name=f'{p}_ff_dropout2')(layers.Dense(64, name=f'{p}_ff2')(f))
    return layers.LayerNormalization(epsilon=1e-6, name=f'{p}_ff_ln')(layers.Add()([x, f]))

def light_block(x, i):                            # ST block (Step 10D `_st_transformer_block`): 1 Dropout layer
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.2, name=f'mha_{i}')(x, x)
    x = layers.LayerNormalization(epsilon=1e-6, name=f'attn_ln_{i}')(layers.Add(name=f'attn_add_{i}')([x, a]))
    f = layers.Dense(128, activation=tf.nn.gelu, name=f'ff1_{i}')(x)
    f = layers.Dense(64, name=f'ff2_{i}')(layers.Dropout(0.2, name=f'ff_dropout_{i}')(f))
    return layers.LayerNormalization(epsilon=1e-6, name=f'ff_ln_{i}')(layers.Add(name=f'ff_add_{i}')([x, f]))

def _trunk(inp):
    return PositionalEncoding(168, 64, name='positional_encoding')(layers.Dense(64, name='input_projection')(inp))

def build_st(target, agg='GAP'):                  # Single-task (existing ST-GAP / ST-LSTA arms)
    inp = keras.Input((168, 8)); x = light_block(light_block(_trunk(inp), 1), 2)
    x = layers.GlobalAveragePooling1D()(x) if agg == 'GAP' else LastState(name='lsta')(x)
    x = layers.Dropout(0.2, name='head_dropout')(layers.Dense(64, activation=tf.nn.gelu, name='dense_head')(x))
    return keras.Model(inp, layers.Dense(1, name=target)(x), name=f'ST_{agg}_{target}')

def build_mtl(agg='LSTA', block='light'):         # Multi-task: shared encoder + two heads
    inp = keras.Input((168, 8)); x = _trunk(inp)
    if block == 'light': x = light_block(light_block(x, 1), 2)
    else:                x = heavy_block(heavy_block(x, 'block1'), 'block2')
    x = layers.GlobalAveragePooling1D()(x) if agg == 'GAP' else LastState(name='lsta')(x)
    x = layers.Dropout(0.2, name='shared_dropout')(layers.Dense(64, activation=tf.keras.activations.gelu, name='shared_dense')(x))
    return keras.Model(inp, [layers.Dense(1, name='load')(x), layers.Dense(1, name='solar')(x)], name=f'MTL_{agg}_{block}')

def n_dropout(m): return sum(isinstance(l, layers.Dropout) for l in m.layers)

AUDIT = []
for name, m, p_exp, d_exp in [('ST-GAP', build_st('load', 'GAP'), 71745, 3), ('ST-LSTA', build_st('load', 'LSTA'), 71745, 3),
                              ('MTL-GAP light', build_mtl('GAP', 'light'), 71810, 3), ('MTL-LSTA light', build_mtl('LSTA', 'light'), 71810, 3),
                              ('MTL-GAP heavy (original)', build_mtl('GAP', 'heavy'), 71810, 7), ('MTL-LSTA heavy (original)', build_mtl('LSTA', 'heavy'), 71810, 7)]:
    AUDIT.append([name, m.count_params(), n_dropout(m)])
    assert m.count_params() == p_exp and n_dropout(m) == d_exp, (name, m.count_params(), n_dropout(m))
tf.keras.backend.clear_session()
print(pd.DataFrame(AUDIT, columns=['model', 'params', 'Dropout layers']).to_string(index=False)); print('ARCHITECTURE AUDIT: PASS')

def set_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def train_one(arm, h, seed, d):
    """arm in {'MTL-GAP','MTL-LSTA','ST-GAP','ST-LSTA'}; ST arms train two single-target networks."""
    agg = arm.split('-')[1]; preds = {}; epochs = {}
    jobs = [('both', None)] if arm.startswith('MTL') else [('load', 0), ('solar', 1)]
    for tag, i in jobs:
        tf.keras.backend.clear_session(); set_seeds(seed)
        if arm.startswith('MTL'):
            m = build_mtl(agg, 'light'); ytr = [d['y_train'][:, 0], d['y_train'][:, 1]]; yva = [d['y_val'][:, 0], d['y_val'][:, 1]]
            m.compile(optimizer=tf.keras.optimizers.Adam(LR), loss=['mse', 'mse'], loss_weights=[2.0, 1.0])
        else:
            m = build_st(tag, agg); ytr = d['y_train'][:, i]; yva = d['y_val'][:, i]
            m.compile(optimizer=tf.keras.optimizers.Adam(LR), loss='mse')
        es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)
        hist = m.fit(d['X_train'], ytr, validation_data=(d['X_val'], yva), epochs=MAX_EPOCHS, batch_size=BATCH, shuffle=True, callbacks=[es], verbose=0)
        p = m.predict(d['X_test'], batch_size=256, verbose=0)
        if isinstance(p, (list, tuple)): preds['load'], preds['solar'] = np.ravel(p[0]), np.ravel(p[1])
        else: preds[tag] = np.ravel(p)
        epochs[tag] = (int(np.argmin(hist.history['val_loss'])) + 1, len(hist.history['val_loss']))
        jd = OUT / 'runs' / f'h{h:02d}_seed{seed:03d}_{arm}'
        m.save_weights(str(guard(jd / f'{tag}.weights.h5')))
        pd.DataFrame(hist.history).to_csv(guard(jd / f'{tag}_history.csv'), index_label='epoch')
    pm = np.c_[preds['load'], preds['solar']].astype(np.float64) * TARGET_SCALE + TARGET_MEAN
    return pm, epochs
print('Training functions ready.')


In [ ]:
# ===================== E3 — RUN (resumable; re-run after any disconnect) =====================
ARMS = ['MTL-GAP', 'MTL-LSTA'] + (['ST-GAP', 'ST-LSTA'] if RUN_ST_ARMS else [])
for h in HORIZONS:
    d = DATA[h]; y = d['y_test'].astype(np.float64) * TARGET_SCALE + TARGET_MEAN
    for seed in SEEDS:
        for arm in ARMS:
            jd = OUT / 'runs' / f'h{h:02d}_seed{seed:03d}_{arm}'; done = jd / 'COMPLETED.json'
            if done.exists(): continue
            t0 = time.time(); pm, ep = train_one(arm, h, seed, d)
            np.savez_compressed(guard(jd / 'test_predictions_MW.npz'), pred_MW=pm, y_true_MW=y)
            rec = dict(h=h, seed=seed, arm=arm, block='light', rmse_load=rmse(pm[:, 0] - y[:, 0]), rmse_solar=rmse(pm[:, 1] - y[:, 1]),
                       epochs={k: list(v) for k, v in ep.items()}, minutes=(time.time() - t0) / 60, env=ENV)
            json.dump(rec, open(guard(done), 'w'), indent=1)
            print(f"h={h:2d} seed={seed:3d} {arm:9s} load {rec['rmse_load']:8.2f}  solar {rec['rmse_solar']:8.2f} MW  "
                  f"best/run epochs {rec['epochs']}  {rec['minutes']:.1f} min")
print('ALL RUNS COMPLETE')


In [ ]:
# ===================== E4 — OPTIONAL (run before E5): cross-version reproducibility probe (2 short trainings) =====================
# Retrains ST-solar (light block) and LSTM at h=1, seed 1 in THIS environment and compares with the frozen per-seed RMSE.
# Documents whether fixed-seed Transformer training reproduces across software versions (paper, reproducibility statement).
FROZEN = np.load(BENCH / 'Reliability_V6/degradation_results/tensorflow_models/tensorflow_reliability_clean_predictions.npz')
d = DATA[1]; y = d['y_test'].astype(np.float64) * TARGET_SCALE + TARGET_MEAN
def build_lstm():
    inp = tf.keras.Input((168, 8))
    x = layers.Dropout(0.2)(layers.LSTM(64, return_sequences=True)(inp)); x = layers.Dropout(0.2)(layers.LSTM(32)(x))
    x = layers.Dropout(0.2)(layers.Dense(64, activation=tf.keras.activations.gelu)(x))
    m = tf.keras.Model(inp, [layers.Dense(1, name='load')(x), layers.Dense(1, name='solar')(x)]); assert m.count_params() == 33346; return m
rows = []
tf.keras.backend.clear_session(); set_seeds(1); m = build_st('solar', 'GAP'); m.compile(tf.keras.optimizers.Adam(LR), 'mse')
m.fit(d['X_train'], d['y_train'][:, 1], validation_data=(d['X_val'], d['y_val'][:, 1]), epochs=MAX_EPOCHS, batch_size=BATCH,
      callbacks=[tf.keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)], verbose=0)
p = np.ravel(m.predict(d['X_test'], verbose=0)) * TARGET_SCALE[1] + TARGET_MEAN[1]
rows.append(['ST solar, h=1, seed 1', rmse(FROZEN['ST_Transformer_seed_1_pred_mw'][:, 1] - y[:, 1]), rmse(p - y[:, 1])])
tf.keras.backend.clear_session(); set_seeds(1); m = build_lstm(); m.compile(tf.keras.optimizers.Adam(LR), ['mse', 'mse'])
m.fit(d['X_train'], [d['y_train'][:, 0], d['y_train'][:, 1]], validation_data=(d['X_val'], [d['y_val'][:, 0], d['y_val'][:, 1]]),
      epochs=MAX_EPOCHS, batch_size=BATCH, callbacks=[tf.keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)], verbose=0)
p = m.predict(d['X_test'], verbose=0); p = np.c_[np.ravel(p[0]), np.ravel(p[1])] * TARGET_SCALE + TARGET_MEAN
rows.append(['LSTM load, h=1, seed 1', rmse(FROZEN['LSTM_seed_1_pred_mw'][:, 0] - y[:, 0]), rmse(p[:, 0] - y[:, 0])])
P = pd.DataFrame(rows, columns=['run', 'frozen RMSE (original software)', f'rerun RMSE (TF {tf.__version__})']); P['difference'] = P.iloc[:, 2] - P.iloc[:, 1]
print(P.round(2).to_string(index=False)); P.to_csv(guard(OUT / 'repro_probe.csv'), index=False)


In [ ]:
# ===================== E5 — SUMMARY + BUNDLE (includes the E4 probe if it was run) =====================
R = pd.DataFrame([json.load(open(p)) for p in sorted((OUT / 'runs').glob('*/COMPLETED.json'))])
print('runs completed:', len(R))
S = R.groupby(['arm', 'h'])[['rmse_load', 'rmse_solar']].agg(['mean', 'std']).round(3)
print(S.to_string())
capped = R.epochs.apply(lambda e: any(v[1] >= MAX_EPOCHS for v in e.values())).sum()
print(f'runs that reached the {MAX_EPOCHS}-epoch limit: {capped}')
R.drop(columns=['env']).to_csv(guard(OUT / 'stageE_runs.csv'), index=False)
json.dump(ENV, open(guard(OUT / 'environment.json'), 'w'), indent=1)
zp = guard(OUT / 'StageE_bundle.zip')
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT / 'stageE_runs.csv', 'stageE_runs.csv'); zf.write(OUT / 'environment.json', 'environment.json')
    for p in (OUT / 'runs').glob('*/*'):
        if p.suffix in ('.csv', '.json', '.npz'): zf.write(p, f'runs/{p.parent.name}/{p.name}')
    if (OUT / 'repro_probe.csv').exists(): zf.write(OUT / 'repro_probe.csv', 'repro_probe.csv')
print(f'Bundle: {zp} ({zp.stat().st_size/1e6:.1f} MB)')
